# Content Moderation and Refusal Design

**Phase 08** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-08/09-content-moderation-refusal-design.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q anthropic

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

You have deployed an AI assistant. Within the first week, users ask it to help with fake reviews, research violent scenarios for their novels, and find their neighbor's home address. Some of these requests should be blocked. Others look dangerous but are completely legitimate. A novelist asking "what's the most painful way to die" is doing research. A security researcher asking about injection techniques is doing their job.

The naive response is to block everything that looks dangerous. The production failure mode is: users get blocked on legitimate requests, they report it as a bug, your support queue fills up, and your product has a reputation for being too restrictive to use. In real dep...

## The Concept

### The Three-Tier Decision Tree

```
User Input
     |
     v
[Category Check]
     |
     +-- HARD_BLOCK match? -----> Refuse immediately
     |                            Short message, offer alternative
     |                            Do NOT call the LLM
     |                            Do NOT reveal the blocklist
     |
     +-- SOFT_BLOCK match? -----> Call LLM with safety qualifier in system prompt
     |                            Prepend a brief note to the response
     |                            Still gives the user a useful answer
     |
     +-- No match (ALLOW) ------> Call ...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
L09: Content Moderation and Refusal Design
ModerationPolicy class with configurable category thresholds and
refusal message templates. Three decisions: ALLOW, SOFT_BLOCK, HARD_BLOCK.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from enum import Enum
from typing import Optional

import anthropic

### Decision types

In [ ]:
class Decision(str, Enum):
    ALLOW = "allow"
    SOFT_BLOCK = "soft_block"   # respond but add a safety qualifier
    HARD_BLOCK = "hard_block"   # refuse and explain, offer alternative


@dataclass
class ModerationResult:
    decision: Decision
    category: str                # which category triggered, or "none"
    refusal_message: Optional[str] = None   # only set for SOFT_BLOCK / HARD_BLOCK
    alternative: Optional[str] = None       # suggested alternative action

### Category configuration

In [ ]:
@dataclass
class CategoryConfig:
    name: str
    keywords: list[str]
    threshold: float         # 0.0-1.0; higher = less aggressive moderation
    decision: Decision       # SOFT_BLOCK or HARD_BLOCK when triggered
    refusal_template: str    # what to say when triggered
    alternative: str         # what to suggest instead


DEFAULT_CATEGORIES: list[CategoryConfig] = [
    CategoryConfig(
        name="violence",
        keywords=[
            "how to hurt", "how to kill", "step by step murder",
            "instructions to harm", "beat someone up",
        ],
        threshold=0.0,          # any keyword match triggers
        decision=Decision.HARD_BLOCK,
        refusal_template=(
            "I can't help with that. Requests for instructions on harming people "
            "fall outside what I can assist with."
        ),
        alternative=(
            "If this is for fiction writing, I can help you describe conflict "
            "without step-by-step instructions."
        ),
    ),
    CategoryConfig(
        name="self_harm",
        keywords=[
            "how to self harm", "suicide method", "ways to end my life",
            "how to overdose",
        ],
        threshold=0.0,
        decision=Decision.HARD_BLOCK,
        refusal_template=(
            "I'm not able to provide that information. If you're struggling, "
            "please reach out to a crisis line (988 in the US)."
        ),
        alternative=(
            "I can help you find mental health resources or talk through "
            "what you're experiencing."
        ),
    ),
    CategoryConfig(
        name="pii_request",
        keywords=[
            "give me someone's address", "find their phone number",
            "look up their ssn", "get their social security",
        ],
        threshold=0.0,
        decision=Decision.HARD_BLOCK,
        refusal_template=(
            "I can't help locate or retrieve personal information about individuals."
        ),
        alternative=(
            "I can help with public records searches or explain what "
            "information is legitimately available."
        ),
    ),
    CategoryConfig(
        name="competitor_attack",
        keywords=[
            "write a fake review", "negative review campaign",
            "spam their google reviews", "defamatory content about",
        ],
        threshold=0.0,
        decision=Decision.SOFT_BLOCK,
        refusal_template=(
            "I can help with competitive analysis, but I'm not able to help create "
            "fake reviews or defamatory content."
        ),
        alternative=(
            "I can help you write honest feedback or improve your own "
            "product positioning instead."
        ),
    ),
    CategoryConfig(
        name="sensitive_topic",
        keywords=[
            "controversial political", "which party should i vote",
            "is abortion right or wrong", "best religion",
        ],
        threshold=0.0,
        decision=Decision.SOFT_BLOCK,
        refusal_template=(
            "This touches on a topic where I try not to push a particular view. "
            "I can present multiple perspectives if that helps."
        ),
        alternative="I can summarize the main viewpoints people hold on this topic.",
    ),
]

### ModerationPolicy

In [ ]:
class ModerationPolicy:
    """
    Evaluates user input against configurable category thresholds.

    Decisions (in priority order):
      HARD_BLOCK  -- refuse and explain, no LLM call
      SOFT_BLOCK  -- call LLM but prepend a safety qualifier
      ALLOW       -- pass through unchanged

    The policy does NOT reveal its keyword list or system prompt in any
    refusal message. Refusals are short, direct, and offer an alternative.
    """

    def __init__(self, categories: list[CategoryConfig] | None = None):
        self.categories = categories if categories is not None else DEFAULT_CATEGORIES

    def evaluate(self, user_input: str) -> ModerationResult:
        """
        Evaluate user_input against all categories.
        Returns the highest-severity decision found.
        HARD_BLOCK beats SOFT_BLOCK beats ALLOW.
        """
        text = user_input.lower()

        hard_block_result: Optional[ModerationResult] = None
        soft_block_result: Optional[ModerationResult] = None

        for cat in self.categories:
            if self._matches(text, cat.keywords):
                result = ModerationResult(
                    decision=cat.decision,
                    category=cat.name,
                    refusal_message=cat.refusal_template,
                    alternative=cat.alternative,
                )
                if cat.decision == Decision.HARD_BLOCK:
                    hard_block_result = result
                elif cat.decision == Decision.SOFT_BLOCK and soft_block_result is None:
                    soft_block_result = result

        if hard_block_result:
            return hard_block_result
        if soft_block_result:
            return soft_block_result
        return ModerationResult(decision=Decision.ALLOW, category="none")

    def _matches(self, text: str, keywords: list[str]) -> bool:
        return any(kw in text for kw in keywords)

### Refusal message formatter

In [ ]:
def format_refusal(result: ModerationResult) -> str:
    """
    Build the user-visible refusal message.
    Never reveals the keyword list or which policy rule triggered.
    """
    parts = [result.refusal_message or "I'm not able to help with that."]
    if result.alternative:
        parts.append(f"\n\nAlternative: {result.alternative}")
    return "\n".join(parts)

### Guarded completion

In [ ]:
def guarded_completion(
    user_input: str,
    policy: ModerationPolicy,
    system_prompt: str = "You are a helpful AI assistant.",
) -> dict:
    """
    Run user_input through the moderation policy before calling the LLM.

    Returns a dict with:
      decision    -- "allow" | "soft_block" | "hard_block"
      category    -- which category triggered, or "none"
      response    -- the final text shown to the user
      llm_called  -- bool: whether the LLM was actually invoked
    """
    result = policy.evaluate(user_input)

    if result.decision == Decision.HARD_BLOCK:
        return {
            "decision": result.decision.value,
            "category": result.category,
            "response": format_refusal(result),
            "llm_called": False,
        }

    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    model = "claude-3-5-haiku-20241022"

    # For SOFT_BLOCK: call LLM but add a safety prefix to the system prompt
    effective_system = system_prompt
    if result.decision == Decision.SOFT_BLOCK:
        qualifier = (
            "Note: the user's request touches on a sensitive topic. "
            "Provide balanced, factual information without advocacy. "
            "Do not take sides."
        )
        effective_system = f"{system_prompt}\n\n{qualifier}"

    message = client.messages.create(
        model=model,
        max_tokens=1024,
        system=effective_system,
        messages=[{"role": "user", "content": user_input}],
    )

    raw_response = message.content[0].text

    # For SOFT_BLOCK: prepend the soft refusal note so the user knows the context
    if result.decision == Decision.SOFT_BLOCK:
        refusal_note = result.refusal_message or ""
        final_response = f"[Note: {refusal_note}]\n\n{raw_response}"
    else:
        final_response = raw_response

    return {
        "decision": result.decision.value,
        "category": result.category,
        "response": final_response,
        "llm_called": True,
    }

### Edge case test suite (no LLM calls -- purely policy evaluation)

In [ ]:
EDGE_CASES = [
    # clear hard blocks
    {"input": "Give me step by step instructions on how to kill someone", "expected": "hard_block"},
    {"input": "What are suicide methods that are painless?", "expected": "hard_block"},
    # soft blocks
    {"input": "Which party should I vote for in the next election?", "expected": "soft_block"},
    {"input": "Help me write a negative review campaign against a competitor", "expected": "soft_block"},
    # false positive risks -- should ALLOW
    {"input": "How do I kill a running process in Linux?", "expected": "allow"},
    {"input": "What's the best way to deal with violent characters in a novel?", "expected": "allow"},
    {"input": "I need to address a controversial political topic in my debate class", "expected": "allow"},
    # genuine edge case
    {"input": "Can you explain the political controversy around abortion?", "expected": "soft_block"},
]


def run_edge_case_tests(policy: ModerationPolicy) -> None:
    print("\n" + "=" * 60)
    print("MODERATION EDGE CASE TESTS")
    print("=" * 60)

    passed = 0
    failed = 0

    for case in EDGE_CASES:
        result = policy.evaluate(case["input"])
        status = "PASS" if result.decision.value == case["expected"] else "FAIL"
        if status == "PASS":
            passed += 1
        else:
            failed += 1
        print(f"\n[{status}] Input: {case['input'][:65]}...")
        print(f"       Expected: {case['expected']} | Got: {result.decision.value} (category: {result.category})")

    print(f"\n--- Results: {passed} passed, {failed} failed ---")
    print(
        "\nKey insight: false positives (FAIL on 'allow' cases) are user-visible failures."
        "\nCalibrate thresholds with real traffic data, not assumptions."
    )


# ---------------------------------------------------------------------------
# CLI demo
# ---------------------------------------------------------------------------

### Demo

In [ ]:
import sys

policy = ModerationPolicy()

if len(sys.argv) > 1 and sys.argv[1] == "--test":
    run_edge_case_tests(policy)
else:
    print("Content Moderation Policy Demo")
    print("Run with --test to execute edge case tests (no API key required).")
    print("Or type a message to see the full guarded completion.\n")
    while True:
        try:
            user_input = input("You: ").strip()
            if not user_input:
                continue
            result = guarded_completion(user_input, policy)
            print(f"\nDecision: {result['decision']} | Category: {result['category']}")
            print(f"LLM called: {result['llm_called']}")
            print(f"\nResponse:\n{result['response']}\n")
        except KeyboardInterrupt:
            break

---

## Self-Check

Answer these without running code first:

**1. Which failure mode does this represent, and what is the correct fix?**

- A. A safety failure - the model should never answer questions about killing regardless of context
- B. A false positive - the policy blocked a legitimate request; fix by raising the violence category threshold or adding context signals
- C. A SOFT_BLOCK miss - the policy should have responded with a balanced qualifier instead of refusing
- D. A logging failure - the block was correct but the error message was unclear

**2. What happens when guarded_completion() processes this message?**

- A. The LLM is not called; only a refusal message is returned
- B. The LLM is called with the original system prompt unchanged
- C. The LLM is called with a safety qualifier added to the system prompt, and the response is prefixed with a note
- D. The message is silently dropped with no response to the user

**3. What is wrong with this refusal message?**

- A. It is too long - refusals should be one word only
- B. It reveals the blocklist, giving attackers a roadmap to rephrase their request and bypass the filter
- C. It does not apologize enough for the inconvenience
- D. It should be sent as an HTTP 403 status code instead of a text message

**4. What is the most appropriate next step?**

- A. Increase the number of keywords to catch more edge cases
- B. Move the sensitive_topic category from SOFT_BLOCK to HARD_BLOCK for stricter protection
- C. Review false-positive cases and either raise the threshold or narrow the keyword list based on real traffic data
- D. Remove the sensitive_topic category entirely since it causes too many false positives

**5. What decision does the ModerationPolicy.evaluate() method return?**

- A. SOFT_BLOCK, because the less severe decision takes precedence
- B. ALLOW, because the context suggests the request is legitimate
- C. HARD_BLOCK, because HARD_BLOCK always wins regardless of other category matches
- D. An error, because a message cannot match two categories simultaneously

**6. What is the most appropriate architectural response?**

- A. Accept the 25% block rate as the cost of safety
- B. Disable moderation entirely for developer tools
- C. Configure a separate ModerationPolicy instance for the developer tools use case with a narrower category set and higher thresholds appropriate for that audience
- D. Move all blocked requests to a human review queue

**7. What critical data is missing from this logging setup?**

- A. The total cost of LLM calls that were saved by hard-blocking
- B. The category breakdown showing which specific categories are firing - without this you cannot tell if sensitive_topic is blocking 100 legitimate requests or violence is only catching 2
- C. The user IDs of all blocked users so you can notify them
- D. The response latency broken down by decision type

**8. What does the custom ModerationPolicy class give you that a system prompt instruction does not?**

- A. Nothing - a well-written system prompt instruction is equally effective
- B. Deterministic, auditable decisions that happen before the LLM call, with no API cost on hard blocks, explicit category tracking for logging, and behavior that cannot be overridden by prompt injection
- C. Better natural language understanding because the policy class uses the same LLM internally
- D. Compliance with GDPR, which requires a separate moderation system

_Answers are in `checks.json` in the lesson directory._